### DOCUMENTACIÓN DEL CÓDIGO (NAVEGACIÓN EN CAMPUS CON A*)

### Importación de Librerías

Este bloque importa las librerías necesarias para la implementación del algoritmo A*:

- **`heapq`**: Implementa una cola de prioridad (heap) necesaria para mantener la frontera ordenada por valores f(n) en A*

- **`math`**: Proporciona funciones matemáticas, especialmente `sqrt()` para calcular la distancia euclídea en la heurística

In [ ]:
import heapq
import math

### BLOQUE 2: Definición del Mapa del Campus

### Configuración del Mapa

Define las dimensiones y estructura del campus universitario:

- **`GRID_W`**: Ancho de la grilla: x va de 0 a 16

- **`GRID_H`**: Alto de la grilla: y va de 0 a 15

In [ ]:
GRID_W = 17
GRID_H = 16

## Estructura del Mapa

- **`CAMPUS_MAP`**: Matriz del Campus

### Dimensiones
- **Grilla 2D**: 17 columnas × 16 filas
- **Sistema de coordenadas**: `(x, y)` donde:
  - `x ∈ [0, 16]` (columna, de izquierda a derecha)
  - `y ∈ [0, 15]` (fila, de arriba hacia abajo)

### Simbología del Mapa

| Símbolo | Significado | Transitabilidad |
|---------|-------------|-----------------|
| `" "` (espacio) | Pasillo / Zona libre | Transitable |
| `"#"` | Pared / Obstáculo | No transitable |
| `"B30"` - `"B38"` | Edificios del campus | Transitable (entrada) |
| `"C"` | Cafetería | Transitable |
| `"B"` | Biblioteca | Transitable |

### Interpretación del Mapa

- Cada **fila** de la matriz `CAMPUS_MAP` representa un valor de `y` (de arriba hacia abajo)
- Cada **columna** dentro de una fila representa un valor de `x` (de izquierda a derecha)
- Los edificios y instalaciones están ubicados en posiciones específicas dentro de esta grilla, marcados con sus respectivas etiquetas
- Las celdas con `"#"` representan obstáculos físicos que no pueden ser atravesados por el agente
- Las celdas con `" "` (espacio) representan áreas transitables (pasillos, zonas comunes)
- Las etiquetas de edificios (`"B30"`, `"B31"`, etc.) ocupan una celda completa y representan el punto de entrada al edificio a nivel de calle (piso 1)

In [ ]:
CAMPUS_MAP = [
    [" ", "#", " ",  " ", "#",  " ",  " ", "#", " ",  " ", "#", " ",  " ", "#", " ",  " ", " "], 
    [" ", "#", " ",  " ", "#",  " ",  " ", "#", " ",  " ", "#", " ",  " ", "#", " ",  " ", " "], 
    [" ", " ", "B30"," ", " ", "B31"," ", " ", "B32"," ", " ", "B33"," ", " ", "B34"," ", " "],  
    [" ", "#", " ",  "#", "#", " ",  "#", " ", "#",  "#", " ", "#",  " ", "#", " ",  "#", " "],  
    [" ", "#", " ",  " ", " ", " ",  "#", " ", "#",  " ", " ", "#",  " ", "#", " ",  "#", " "],  
    [" ", " ", " ",  "#", " ", " ",  " ", " ", " ",  "#", " ", " ",  " ", " ", " ",  "#", " "],  
    [" ", "#", " ",  " ", "B35"," ",  "#", " ", "B36"," ", "#", " ",  "B37"," ", "#", " ", "B38"], 
    [" ", "#", "#",  " ", "#", " ",  "#", " ", "#",  " ", "#", " ",  "#", " ", "#",  " ", " "],  
    [" ", " ", " ",  " ", "#", " ",  " ", " ", "#",  " ", " ", " ",  "#", " ", " ",  " ", " "],  
    ["#", "#", " ",  "#", "#", " ",  "#", " ", " ",  " ", "#", " ",  "#", " ", "#",  "#", " "],  
    [" ", " ", " ",  " ", "#", " ",  "C", " ", "#",  " ", "#", " ",  " ", " ", "B",  " ", " "],   
    [" ", "#", "#",  " ", "#", " ",  "#", " ", "#",  " ", "#", " ",  "#", " ", "#",  " ", " "],  
    [" ", " ", " ",  " ", " ", " ",  " ", " ", " ",  " ", " ", " ",  " ", " ", " ",  " ", " "],  
    [" ", "#", " ",  "#", " ", "#",  " ", "#", " ",  "#", " ", "#",  " ", "#", " ",  "#", " "], 
    [" ", "#", " ",  " ", " ", " ",  " ", " ", " ",  " ", " ", " ",  " ", " ", " ",  "#", " "],  
    [" ", " ", " ",  "#", "#", "#",  " ", "#", "#",  "#", " ", "#",  "#", "#", " ",  " ", " "],  
]

## BLOQUE 3: Coordenadas y Configuración de Edificios

### Mapeo de Edificios y sus Coordenadas

- Define las ubicaciones exactas de cada edificio en el campus.

- **`BUILDING_COORDS`**: Diccionario que mapea cada edificio a su coordenada (x, y)

In [ ]:
BUILDING_COORDS = {
    'B30': (2, 2),
    'B31': (5, 2),
    'B32': (8, 2),
    'B33': (11, 2),
    'B34': (14, 2),
    'B35': (4, 6),
    'B36': (8, 6),
    'B37': (12, 6),
    'B38': (16, 6),
    'Cafetería': (6, 10),    
    'Biblioteca': (14, 10),    
}

- **`BUILDINGS_WITH_ELEVATORS`**: Solo estos 5 edificios tienen ascensor (B32, B33, B35, B37, B38)

- **`Edificios sin ascensor`**: B30, B31, B34, B36, Cafetería y Biblioteca solo tienen escaleras

- **`BUILDINGS`**: Conjunto con todos los nombres de edificios del campus

In [ ]:
BUILDINGS_WITH_ELEVATORS = {'B32', 'B33', 'B35', 'B37', 'B38'}
BUILDINGS = set(BUILDING_COORDS.keys())

## BLOQUE 4: Acciones y Costos

### Definición de Movimientos y Costos

Define las acciones posibles y sus costos asociados.

### Explicación de Costos:

- **`Movimiento horizontal (arriba/abajo/izquierda/derecha)`**: Costo = 1 por cada casilla

- **`Ascensor subir/bajar`**: Costo = 3 por cada piso
- **`Escaleras subir`**: Costo p donde p = número de pisos a subir
    - Subir 1 piso: 1 = 1
    - Subir 2 pisos: 2 = 2
- **`Escaleras bajar`**: Costo p donde p = número de pisos a bajar
    - Bajar 1 piso: 1 = 1
    - Bajar 2 pisos: 2 = 2

## RESTRICCIÓN IMPORTANTE:
Los movimientos horizontales (arriba/abajo/izquierda/derecha) SOLO están permitidos en el piso 1. Si estás en piso 2 o 3, primero debes bajar al piso 1 para moverte horizontalmente.



In [ ]:
PLAN_MOVES = {
    'arriba':    (0, -1),
    'abajo':     (0, 1),
    'izquierda': (-1, 0),
    'derecha':    (1, 0),
}

In [ ]:
ACTION_COSTS = {
    'mover': 1,
    'subir_ascensor': 3,
    'bajar_ascensor': 3,
    'subir_escaleras': None,
    'bajar_escaleras': 5,
}

## BLOQUE 5: Funciones Auxiliares

### Funciones Auxiliares del Mapa

### Función `edificio_from_coord(coord)`:

- **Propósito**: Determinar qué edificio (si existe) está en una coordenada dada

- **Entrada**: Tupla (x, y) con la coordenada

- **Salida**:
    - Nombre del edificio si coincide con alguna coordenada conocida
    - 'Pasillo' si no hay ningún edificio en esa posición
- **Ejemplo**: edificio_from_coord((2, 2)) → 'B30'


### Función `is_transitable(coord)`:

- **Propósito**: Verificar si una coordenada es válida para moverse

- **Entrada**: Tupla (x, y) con la coordenada

- **Salida**:
    - True si la posición está dentro del mapa y NO es una pared (#)
    - False si está fuera de límites o es una pared
- **Uso**: Validar movimientos antes de generar sucesores

In [ ]:
def edificio_from_coord(coord):
    for name, c in BUILDING_COORDS.items():
        if coord == c:
            return name
    return 'Pasillo'

In [ ]:
def is_transitable(coord):
    x, y = coord
    if 0 <= x < GRID_W and 0 <= y < GRID_H:
        cell = CAMPUS_MAP[y][x]
        return cell != '#'
    return False

## BLOQUE 6: Estructura de Datos - Clase Node

### Clase Node: Representa un nodo en el árbol de búsqueda


## Atributos del Nodo

- `state`: Diccionario con:

    - `'edificio'`: Nombre del edificio o `'Pasillo'`
    - `'piso'`: Número del piso (1, 2 o 3)
    - `'coordenada'`: Tupla `(x, y)`

- `parent`: Referencia al nodo padre (para reconstruir el camino).

- `action`: Acción que se tomó para llegar a este nodo (ej: `'derecha'`, `'subir_ascensor'`).

- `path_cost (g(n))`: Costo real acumulado desde el nodo inicial hasta este nodo.

- `f_value (f(n))`: Valor de evaluación:

- `g(n)`: Costo del camino hasta este nodo.

- `h(n)`: Estimación heurística del costo restante hasta el objetivo.

## Métodos especiales

- `__lt__`: Permite comparar nodos por su `f_value` (necesario para `heapq`).

- `__repr__`: Representación legible del nodo para debugging.


In [ ]:
class Node:
    def __init__(self, state, parent=None, action=None, path_cost=0, f_value=float('inf')):
        self.state = state
        self.parent = parent
        self.action = action  # Acción que llevó a este estado
        self.path_cost = path_cost  # Costo del path g(n)
        self.f_value = f_value  # valor f(n) = g(n) + h(n)

    def __lt__(self, other):
        return self.f_value < other.f_value  # comparación para la organización de prioridad

    def __repr__(self):
        return f"Node({self.state}, g={self.path_cost:.1f}, f={self.f_value:.1f})"

## BLOQUE 7: Clase Problem - Define el Problema de Búsqueda

**Clase Problem**: Define el problema de búsqueda

- **`Método is_goal(state)`**: Verifica si un estado corresponde al objetivo.

  - **Propósito**: Determinar si el estado actual es el estado objetivo.
  - **Condición**: Debe coincidir tanto la coordenada como el piso.

  - **Ejemplo**:
    - Estado: `{'edificio': 'B30', 'piso': 2, 'coordenada': (2, 2)}`
    - Goal: `{'edificio': 'B30', 'piso': 2, 'coordenada': (2, 2)}`
    - Resultado: `True`

- **`Método h(state)`**: Función heurística que estima el costo restante hasta el objetivo.

  - **Tipo**: Distancia euclídea en el plano 2D.

  - **Fórmula**:
    ```
    h(n) = sqrt((x2 - x1)^2 + (y2 - y1)^2)
    ```

  - **Características**:
    - **Admisible**: Nunca sobreestima el costo real (la línea recta es el camino más corto).
    - **Ignora la diferencia de pisos**: Solo considera la distancia horizontal.
    - **Optimista**: Asume que se puede ir en línea recta (sin obstáculos).

- **Por qué ignora el piso**:
  - El algoritmo descubre el costo real de cambiar de piso mediante `g(n)`.
  - Mantiene la heurística simple y eficiente.
  - Sigue siendo admisible (requisito para que A* sea óptimo).



In [ ]:
class Problem:
    def __init__(self, initial, goal):
        self.initial = initial
        self.goal = goal

    def is_goal(self, state):
        return state['coordenada'] == self.goal['coordenada'] and state['piso'] == self.goal['piso']

    def h(self, state):
        x1, y1 = state['coordenada']
        x2, y2 = self.goal['coordenada']
        return math.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)

## BLOQUE 8: Función de Evaluación f(n)

### Función f(n) = g(n) + h(n)

- **`f(n)`**: Función de evaluación que combina el costo real y la estimación heurística.

  - **Definición**:
    ```
    f(n) = g(n) + h(n)
    ```

- **`g(n) = node.path_cost`**: Costo real acumulado desde el nodo inicial hasta el nodo actual.

  - Costo exacto, no es una estimación.
  - Aumenta a medida que el nodo se aleja del estado inicial.

- **`h(n) = problem.h(node.state)`**: Estimación heurística del costo desde el nodo actual hasta el objetivo.

  - En este caso: distancia euclídea.
  - Disminuye a medida que el nodo se acerca al objetivo.

- **Significado de `f(n)`**:

  - Representa la estimación del costo total del camino más barato que pasa por el nodo `n`.
  - El algoritmo A* expande primero los nodos con menor `f(n)`.
  - Garantiza encontrar el camino óptimo si `h(n)` es admisible.

- **Ejemplo**:

  - Nodo en `(5, 5)`, objetivo en `(10, 10)`, `path_cost = 12`.

  - Cálculo de la heurística:
    ```
    h(n) = sqrt((10 - 5)^2 + (10 - 5)^2) = sqrt(50) ≈ 7.07
    ```

  - Cálculo de la función de evaluación:
    ```
    f(n) = 12 + 7.07 = 19.07
    ```


In [ ]:
def f(node, problem):
    return node.path_cost + problem.h(node.state)

## BLOQUE 9: Función Expand - Generar Sucesores

### Función expand(): Genera todos los sucesores válidos de un nodo

Esta es una de las funciones más importantes del algoritmo A*. Genera todos los estados alcanzables desde el estado actual aplicando las acciones válidas.

- **Estructura general**: Describe el proceso para generar los nodos sucesores a partir del estado actual.

  - Extrae información del estado actual (`edificio`, `piso`, `coordenada`).
  
  - Aplica los tres tipos de acciones posibles:
    - **Movimientos en el plano**: Solo disponibles si `piso == 1`.
    - **Ascensor**: Solo disponible en edificios que cuentan con ascensor.
    - **Escaleras**: Disponibles en cualquier edificio excepto en pasillos.

  - Para cada acción válida, crea un nodo hijo.

  - Calcula `f(n)` para cada nodo hijo.

  - Retorna la lista de sucesores.

---

- **Parte 1: Movimientos en el plano**: Permiten desplazamiento horizontal entre coordenadas.

  - **Restricción clave**: `if piso == 1`
    - Los movimientos horizontales solo son posibles en el piso 1.
    - Si el agente está en piso 2 o 3, primero debe bajar al piso 1.

  - **Proceso**:
    - Itera sobre las cuatro direcciones: arriba, abajo, izquierda, derecha.
    - Calcula la nueva coordenada sumando el delta `(dx, dy)`.
    - Verifica si la nueva coordenada es transitable.
    - Determina si existe un edificio en la nueva coordenada.
    - Crea un nodo hijo con:
      - Costo incrementado en `1` (`ACTION_COSTS['mover']`).
      - Mismo piso (`1`).
      - Nueva coordenada.
    - Calcula `f(n)` para el nodo hijo.

---

- **Parte 2: Movimientos con ascensor**: Permiten cambiar de piso dentro de ciertos edificios.

  - **Condiciones del ascensor**:
    - Solo disponible en: `B32`, `B33`, `B35`, `B37`, `B38`.
    - Puede subir si `piso < 3` (máximo hasta el piso 3).
    - Puede bajar si `piso > 1` (mínimo hasta el piso 1).
    - Costo: `3` por cada piso (subir o bajar).


---

- **Parte 3: Movimientos con escaleras**: Permiten cambiar de piso en la mayoría de edificios.

  - **Características de las escaleras**:
    - Disponibles en todos los edificios (excepto en `'Pasillo'`).
    - Permiten subir o bajar múltiples pisos en un solo movimiento.



In [ ]:
def expand(node, problem):
    successors = []
    state = node.state
    edificio = state['edificio']
    piso = state['piso']
    coord = state['coordenada']

# 1) Movimientos en el plano (arriba, abajo, izquierda, derecha)
# SOLO se puede mover en el plano si está en el piso 1
    if piso == 1:
        for action_name, delta in PLAN_MOVES.items():
            dx, dy = delta
            new_coord = (coord[0] + dx, coord[1] + dy)
            if is_transitable(new_coord):
                new_edificio = edificio_from_coord(new_coord)
                child = Node(
                    state={'edificio': new_edificio, 'piso': piso, 'coordenada': new_coord},
                    parent=node,
                    action=action_name,
                    path_cost=node.path_cost + ACTION_COSTS['mover']
                )
                child.f_value = f(child, problem)
                successors.append(child)

# 2) Ascensor (solo en edificios con ascensor)
    if edificio in BUILDINGS_WITH_ELEVATORS:
        if piso < 3:
            child = Node(
                state={'edificio': edificio, 'piso': piso + 1, 'coordenada': coord},
                parent=node,
                action='subir_ascensor',
                path_cost=node.path_cost + ACTION_COSTS['subir_ascensor']
            )
            child.f_value = f(child, problem)
            successors.append(child)
        
        if piso > 1:
            child = Node(
                state={'edificio': edificio, 'piso': piso - 1, 'coordenada': coord},
                parent=node,
                action='bajar_ascensor',
                path_cost=node.path_cost + ACTION_COSTS['bajar_ascensor']
            )
            child.f_value = f(child, problem)
            successors.append(child)
            
# 3) Escaleras (disponibles en cualquier edificio)
# Solo usar escaleras si estamos EN un edificio (no en pasillo)
    if edificio != 'Pasillo':
        for p_subir in range(1, 4 - piso + 1):  # Máximo hasta piso 3
            new_piso = piso + p_subir
            if new_piso <= 3:
                child = Node(
                    state={'edificio': edificio, 'piso': new_piso, 'coordenada': coord},
                    parent=node,
                    action=f'subir_escaleras_{p_subir}',
                    path_cost=node.path_cost + (7 + p_subir)
                )
                child.f_value = f(child, problem)
                successors.append(child)
        
        if piso > 1:
            for p_bajar in range(1, piso):
                new_piso = piso - p_bajar
                child = Node(
                    state={'edificio': edificio, 'piso': new_piso, 'coordenada': coord},
                    parent=node,
                    action=f'bajar_escaleras_{p_bajar}',
                    path_cost=node.path_cost + (5 * p_bajar)  # Costo por cada piso bajado
                )
                child.f_value = f(child, problem)
                successors.append(child)

    return successors

## BLOQUE 10: Algoritmo RBFS (Opcional)

- **RBFS (Recursive Best-First Search)**: Variante del algoritmo A* con uso limitado de memoria.

  - **Definición**:
    - Utiliza recursión en lugar de mantener toda la frontera de búsqueda en memoria.
    - Es útil cuando el espacio de búsqueda es muy grande.

  - **Ventajas**:
    - Usa menos memoria que A* estándar.
    - Garantiza encontrar el camino óptimo si `h(n)` es admisible.

  - **Desventajas**:
    - Puede re-expandir nodos (menos eficiente en tiempo).
    - Es más complejo de implementar y entender.

  - **Cuándo usar RBFS vs A***:
    - **A***: Cuando hay suficiente memoria disponible (caso más común).
    - **RBFS**: Cuando el espacio de búsqueda es muy grande y la memoria es limitada.

  - **Recomendación para este ejercicio**:
    - Se recomienda usar A* estándar.


In [ ]:
def recursive_best_first_search(problem):
    initial_node = Node(problem.initial, path_cost=0, f_value=problem.h(problem.initial))
    return rbfs(problem, initial_node, float('inf'))

def rbfs(problem, node, f_limit):
    if problem.is_goal(node.state):
        return node, node.f_value

    successors = expand(node, problem)
    if not successors:
        return None, float('inf')

    heapq.heapify(successors)

    while True:
        best = heapq.heappop(successors)
        if best.f_value > f_limit:
            return None, best.f_value
        alternative = successors[0].f_value if successors else float('inf')
        result, best.f_value = rbfs(problem, best, min(f_limit, alternative))
        if result is not None:
            return result, best.f_value
        heapq.heappush(successors, best)

## BLOQUE 11: Algoritmo A* - Implementación Principal

### Algoritmo A* - Búsqueda del Camino Óptimo

- **Paso 1: Inicialización**: Configura las estructuras necesarias para iniciar el algoritmo A*.

  - **Estructuras de datos**:

    - **`frontier` (frontera)**:
      - Cola de prioridad (`min-heap`) con nodos por explorar.
      - Ordenada por `f(n) = g(n) + h(n)`.
      - Siempre se expande el nodo con menor `f(n)`.

    - **`explored` (explorados)**:
      - Conjunto (`set`) de estados ya visitados.
      - Evita re-explorar el mismo estado.
      - Clave: tupla `(edificio, piso, coordenada)`.

    - **`initial_node`**:
      - Nodo inicial con:
        - `path_cost = 0` (`g(n) = 0` al inicio).
        - `f_value = h(initial_state)` (solo heurística al inicio).

---

- **Paso 2: Bucle principal de A\***: Ejecuta el proceso iterativo de exploración.

  - **Flujo del algoritmo**:

    - **Extraer nodo con menor `f(n)`**:
      - `heapq` devuelve el nodo con menor `f_value`.
      - Garantiza explorar primero los nodos más prometedores.

    - **Verificar si ya fue explorado**:
      - Evita procesar el mismo estado múltiples veces.
      - Mejora la eficiencia.

    - **Verificar si es el objetivo**:
      - Si se encuentra el objetivo, el algoritmo termina.
      - Se garantiza que el camino es óptimo porque los nodos se expanden en orden de `f(n)`.

    - **Expandir nodo**:
      - Genera todos los sucesores válidos.
      - Añade a la frontera los que no han sido explorados.

---

- **Paso 3: Reconstrucción del camino**: Se ejecuta cuando se alcanza el estado objetivo.

  - **Condición**:
    - `if problem.is_goal(state)`

  - **Proceso de reconstrucción**:

    - **Seguir los punteros padre**:
      - Cada nodo apunta a su nodo padre.
      - Se recorre desde el objetivo hacia el nodo inicial.

    - **Recolectar información**:
      - `ruta_estados`: Lista de estados visitados.
      - `ruta_acciones`: Lista de acciones tomadas.
      - `node.path_cost`: Costo total del camino.

    - **Invertir las listas**:
      - La reconstrucción se realiza del objetivo al inicio.
      - `.reverse()` las ordena correctamente (inicio → objetivo).

---

- **Propiedades de A\***: Características teóricas del algoritmo.

  - **Completitud**:
    - Si existe una solución, A* la encontrará.
    - Condición: espacio de búsqueda finito o factor de ramificación finito.

  - **Optimalidad**:
    - A* garantiza encontrar el camino de menor costo.
    - Condición: la heurística `h(n)` debe ser admisible (nunca sobreestima).
    - La heurística euclídea utilizada es admisible.

  - **Complejidad**:
    - **Tiempo**: `O(b^d)` donde `b` es el factor de ramificación y `d` la profundidad.
    - **Espacio**: `O(b^d)` porque mantiene los nodos en memoria.

  - **Por qué funciona**:
    - `f(n) = g(n) + h(n)` combina costo real y estimación.
    - Expande primero los nodos más prometedores (menor `f(n)`).
    - Cuando alcanza el objetivo, garantiza que no existe un camino mejor.


In [ ]:
def astar_search(problem, max_iters=100000):
    initial_node = Node(problem.initial, path_cost=0, f_value=problem.h(problem.initial))
    frontier = []
    heapq.heappush(frontier, initial_node)
    explored = set()

    it = 0
    while frontier and it < max_iters:
        it += 1
        node = heapq.heappop(frontier)
        state = node.state
        key = (state['edificio'], state['piso'], state['coordenada'])
        
        if key in explored:
            continue
        explored.add(key)

        if problem.is_goal(state):
            ruta_estados = []
            ruta_acciones = []
            current = node
            while current is not None:
                ruta_estados.append(current.state)
                if current.action is not None:
                    ruta_acciones.append(current.action)
                current = current.parent
            ruta_estados.reverse()
            ruta_acciones.reverse()
            return ruta_estados, ruta_acciones, node.path_cost

        for succ in expand(node, problem):
            succ_key = (succ.state['edificio'], succ.state['piso'], succ.state['coordenada'])
            if succ_key not in explored:
                heapq.heappush(frontier, succ)

    return None, None, float('inf')

## BLOQUE 12: MUESTRA DE RESULTADOS

- Esta función es para poder imprimir por pantalla la ruta escogida o seguida por el algoritmo.

In [ ]:
def print_resultado(nombre_caso, ruta_estados, ruta_acciones, costo):
    print(f"\n{'='*70}")
    print(f"CASO: {nombre_caso}")
    print(f"{'='*70}")
    
    if ruta_estados is None:
        print("No se encontró ruta.")
        return
    
    print(f"✓ Ruta encontrada con costo total: {costo}")
    print(f"\nAcciones ({len(ruta_acciones)}):")
    print(ruta_acciones)
    
    print(f"\nSecuencia de estados ({len(ruta_estados)}):")
    for i, estado in enumerate(ruta_estados):
        coord = estado['coordenada']
        print(f"  {i+1}. {estado['edificio']:12} | Piso {estado['piso']} | ({coord[0]:2}, {coord[1]:2})")
    print()

## BLOQUE 13: EJEMPLOS DE PRUEBA

- Casos predeterminados para que el usuario vea de forma rápida el funcionamiento del algoritmo.

In [ ]:
def ejemplo_1():
    start = {
        'edificio': edificio_from_coord((0, 0)),
        'piso': 1,
        'coordenada': (0, 0)
    }
    
    goal = {
        'edificio': 'B30',
        'piso': 1,
        'coordenada': BUILDING_COORDS['B30']
    }
    
    problem = Problem(initial=start, goal=goal)
    ruta_estados, ruta_acciones, costo = astar_search(problem)
    print_resultado("Ejemplo 1: (0,0) → B30 Piso 1", ruta_estados, ruta_acciones, costo)


def ejemplo_2():
    start = {
        'edificio': 'B32',
        'piso': 1,
        'coordenada': BUILDING_COORDS['B32']
    }
    
    goal = {
        'edificio': 'B32',
        'piso': 3,
        'coordenada': BUILDING_COORDS['B32']
    }
    
    problem = Problem(initial=start, goal=goal)
    ruta_estados, ruta_acciones, costo = astar_search(problem)
    print_resultado("Ejemplo 2: B32 Piso 1 → B32 Piso 3 (con ascensor)", ruta_estados, ruta_acciones, costo)


def ejemplo_3():
    start = {
        'edificio': 'B30',
        'piso': 1,
        'coordenada': BUILDING_COORDS['B30']
    }
    
    goal = {
        'edificio': 'Biblioteca',
        'piso': 2,
        'coordenada': BUILDING_COORDS['Biblioteca']
    }
    
    problem = Problem(initial=start, goal=goal)
    ruta_estados, ruta_acciones, costo = astar_search(problem)
    print_resultado("Ejemplo 3: B30 Piso 1 → Biblioteca Piso 2", ruta_estados, ruta_acciones, costo)


def ejemplo_4():
    start = {
        'edificio': 'B30',
        'piso': 1,
        'coordenada': BUILDING_COORDS['B30']
    }
    
    goal = {
        'edificio': 'B30',
        'piso': 2,
        'coordenada': BUILDING_COORDS['B30']
    }
    
    problem = Problem(initial=start, goal=goal)
    ruta_estados, ruta_acciones, costo = astar_search(problem)
    print_resultado("Ejemplo 4: B30 Piso 1 → B30 Piso 2 (solo escaleras)", ruta_estados, ruta_acciones, costo)


def ejemplo_5():
    start = {
        'edificio': 'B35',
        'piso': 1,
        'coordenada': BUILDING_COORDS['B35']
    }
    
    goal = {
        'edificio': 'Cafetería',
        'piso': 1,
        'coordenada': BUILDING_COORDS['Cafetería']
    }
    
    problem = Problem(initial=start, goal=goal)
    ruta_estados, ruta_acciones, costo = astar_search(problem)
    print_resultado("Ejemplo 5: B35 Piso 1 → Cafetería Piso 1", ruta_estados, ruta_acciones, costo)

## BLOQUE 14: PRUEBA PERSONALIZADA

- Función que le permitira al usuario escoger el bloque y piso tanto de inicio como de fin.

In [ ]:
def prueba_personalizada(start_edificio, start_piso, goal_edificio, goal_piso):
    start = {
        'edificio': start_edificio,
        'piso': start_piso,
        'coordenada': BUILDING_COORDS.get(start_edificio, (0, 0))  # Si no se encuentra, usar (0,0)
    }
    
    goal = {
        'edificio': goal_edificio,
        'piso': goal_piso,
        'coordenada': BUILDING_COORDS.get(goal_edificio, (0, 0))  # Si no se encuentra, usar (0,0)
    }
    
    problem = Problem(initial=start, goal=goal)
    ruta_estados, ruta_acciones, costo = astar_search(problem)
    print_resultado(f"Prueba Personalizada: {start_edificio} Piso {start_piso} → {goal_edificio} Piso {goal_piso}", ruta_estados, ruta_acciones, costo)

## BLOQUE 15: MENU

- Menu con opciones para que el usuario interactue con el programa.

In [ ]:
def menu():
    print("\n" + "="*70)
    print(" "*20 + "NAVEGACIÓN EN CAMPUS CON A*")
    print("="*70)
    print("Seleccione un ejemplo para ejecutar:")
    print("[1] Ejemplos Predeterminados")
    print("[2] Prueba Personalizada")
    print("[0] Salir")
    
    opcion = input("Ingrese su opción: ")

    if opcion == '1':
        print("\nEjecutando ejemplos predeterminados...")
        ejemplo_1()
        ejemplo_2()
        ejemplo_3()
        ejemplo_4()
        ejemplo_5()
        print("\n" + "="*70)
        print(" "*20 + "PRUEBAS COMPLETADAS")
        print("="*70 + "\n")
    elif opcion == '2':
        print("\nVamos a crear tu propia prueba personalizada.")
        print("Ingresa el punto de inicio:")
        start_edificio = input("Edificio de inicio (B30, B31, B32, B33, B34, B35, B36, B37, B38, Cafetería, Biblioteca): ")
        start_piso = int(input("Piso de inicio (1, 2, o 3): "))
        print("\nIngresa el punto objetivo:")
        goal_edificio = input("Edificio objetivo (B30, B31, B32, B33, B34, B35, B36, B37, B38, Cafetería, Biblioteca): ")
        goal_piso = int(input("Piso objetivo (1, 2, o 3): "))
        
        prueba_personalizada(start_edificio, start_piso, goal_edificio, goal_piso)
        
        print("\n" + "="*70)
        print(" "*20 + "PRUEBA COMPLETADA")
        print("="*70 + "\n")

    elif opcion == '0':
        print("\n¡Hasta luego!")
    else:
        print("\nOpción inválida. Por favor intente de nuevo.")

In [ ]:
if __name__ == "__main__":
    menu()